## 22. Monte Carlo Portfolio Loss Simulation

In [ ]:
# ============================================================
# CELL — Build the per-loan inputs needed for the simulation
# ============================================================
from scipy.stats import norm

# Use the resolved-loan portfolio (el_df) as the simulation universe --
# same population Section 18's stress testing used, for consistency.
N_SIM = 10000

# Per-loan inputs already in memory from Week 1
sim_pd = el_df['pd'].values
sim_lgd = el_df['lgd'].values
sim_ead = el_df['ead'].values
n_loans = len(sim_pd)

print(f"Simulating {N_SIM:,} scenarios across {n_loans:,} loans...")
print(f"rho = {rho}")

# Per-loan default threshold: G(PD), the inverse-normal cutoff such that a
# standard normal asset value falls below it with probability == that loan's PD
sim_threshold = norm.ppf(np.clip(sim_pd, 1e-6, 1 - 1e-6))

print(f"Threshold range: {sim_threshold.min():.3f} to {sim_threshold.max():.3f}")

In [ ]:
# ============================================================
# CELL — Run the Monte Carlo loop
# ============================================================
# Memory-safety note: looping in pure Python over 10,000 sims x ~1.3M loans
# would be far too slow and memory-heavy. Vectorize with numpy instead --
# draw all idiosyncratic factors for one simulation at once as a single
# array operation, rather than looping loan-by-loan.

np.random.seed(42)
portfolio_losses = np.zeros(N_SIM)

sqrt_rho = np.sqrt(rho)
sqrt_1_minus_rho = np.sqrt(1 - rho)

batch_size = 500  # process loans in batches to control memory per simulation
n_batches = int(np.ceil(n_loans / batch_size))

for sim in range(N_SIM):
    Z = np.random.standard_normal()  # one shared economic factor for this scenario

    total_loss = 0.0
    for b in range(n_batches):
        start = b * batch_size
        end = min(start + batch_size, n_loans)

        epsilon = np.random.standard_normal(end - start)  # personal factor, this batch
        asset_value = sqrt_rho * Z + sqrt_1_minus_rho * epsilon

        defaulted = asset_value < sim_threshold[start:end]
        batch_loss = np.where(
            defaulted,
            sim_lgd[start:end] * sim_ead[start:end],
            0.0
        )
        total_loss += batch_loss.sum()

    portfolio_losses[sim] = total_loss

    if (sim + 1) % 1000 == 0:
        print(f"  Completed {sim + 1:,} / {N_SIM:,} simulations...")

print(f"\nSimulation complete. {N_SIM:,} portfolio loss scenarios generated.")

In [ ]:
# ============================================================
# CELL — Plot the full loss distribution: mean, std, skewness, fat tail
# ============================================================
from scipy.stats import skew

mean_loss = portfolio_losses.mean()
std_loss = portfolio_losses.std()
skewness = skew(portfolio_losses)

print("--- Loss Distribution Summary ---")
print(f"Mean portfolio loss:     ${mean_loss:,.0f}")
print(f"Std deviation:           ${std_loss:,.0f}")
print(f"Skewness:                {skewness:.3f}  (positive = long tail toward big losses)")
print(f"Min simulated loss:      ${portfolio_losses.min():,.0f}")
print(f"Max simulated loss:      ${portfolio_losses.max():,.0f}")

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(portfolio_losses, bins=60, color='#378ADD', edgecolor='white', alpha=0.85)
ax.axvline(mean_loss, color='black', linestyle='--', linewidth=1.5, label=f'Mean: ${mean_loss:,.0f}')
ax.set_title(f'Simulated Portfolio Loss Distribution ({N_SIM:,} scenarios, rho={rho})')
ax.set_xlabel('Portfolio Loss ($)')
ax.set_ylabel('Number of Scenarios')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nFor comparison, Week 1's baseline Expected Loss (no correlation modeling): "
      f"${el_df['pd'].values.dot(el_df['lgd'].values * el_df['ead'].values):,.0f}")
print("This should be close to the simulation's mean -- the mean loss shouldn't")
print("change much due to correlation; what changes is the SPREAD and TAIL.")

In [ ]:
# ============================================================
# CELL — Compute VaR and CVaR at 95% and 99.9%
# ============================================================
sorted_losses = np.sort(portfolio_losses)

def compute_var_cvar(sorted_arr, confidence):
    idx = int(np.floor(confidence * len(sorted_arr)))
    idx = min(idx, len(sorted_arr) - 1)
    var = sorted_arr[idx]
    cvar = sorted_arr[idx:].mean()  # average of everything AT or BEYOND the VaR cutoff
    return var, cvar

var_95, cvar_95 = compute_var_cvar(sorted_losses, 0.95)
var_999, cvar_999 = compute_var_cvar(sorted_losses, 0.999)

print("--- VaR / CVaR Table ---")
print(f"{'Metric':<20} {'95% Confidence':>20} {'99.9% Confidence':>20}")
print(f"{'VaR':<20} {'$' + f'{var_95:,.0f}':>20} {'$' + f'{var_999:,.0f}':>20}")
print(f"{'CVaR':<20} {'$' + f'{cvar_95:,.0f}':>20} {'$' + f'{cvar_999:,.0f}':>20}")

print(f"\nInterpretation:")
print(f"95% of the time, portfolio loss will not exceed ${var_95:,.0f}.")
print(f"In the worst 5% of scenarios, average loss is ${cvar_95:,.0f}.")
print(f"99.9% of the time (Basel standard), loss will not exceed ${var_999:,.0f}.")
print(f"In the worst 0.1% of scenarios, average loss is ${cvar_999:,.0f}.")

In [ ]:
# ============================================================
# CELL — Visualize VaR/CVaR cutoffs on the distribution
# ============================================================
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(portfolio_losses, bins=60, color='#378ADD', edgecolor='white', alpha=0.7)
ax.axvline(var_95, color='#BA7517', linestyle='--', linewidth=2, label=f'VaR 95%: ${var_95:,.0f}')
ax.axvline(var_999, color='#A32D2D', linestyle='--', linewidth=2, label=f'VaR 99.9%: ${var_999:,.0f}')
ax.axvline(cvar_999, color='#A32D2D', linestyle=':', linewidth=2, label=f'CVaR 99.9%: ${cvar_999:,.0f}')
ax.set_title('Loss Distribution with VaR / CVaR Cutoffs')
ax.set_xlabel('Portfolio Loss ($)')
ax.set_ylabel('Number of Scenarios')
ax.legend()
plt.tight_layout()
plt.show()